# 06 — Demo en vivo (revisión final, Día 2)

Este es el notebook que se abre frente al evaluador. Guion sugerido:

1. El evaluador da un ID nuevo (o cualquiera del dataset) — o sube un PDF si la feature [PLUS] de Docling está lista.
2. Se corre la celda de consulta con ese ID exacto — el pipeline se ejecuta completo, no hay respuesta precargada (`tests/test_pipeline_live.py` lo prueba de forma automatizada).
3. Se abre `evidence[]` del resultado #1 y se cruza a mano contra el CSV crudo en una segunda ventana — esto es literalmente la 'auditoría de una conexión/evidencia' que pide la rúbrica (Validación técnica, 5 pts).

**Correr la celda de arranque primero, apenas se abra el notebook** — así cuando el evaluador dé el ID, la consulta ya sale en 1-2s.

In [1]:
import sys
sys.path.insert(0, '..')

from saberlink import pipeline

# Llamada de arranque: carga el modelo de embeddings y los índices.
# Se descarta el resultado — solo es para calentar el proceso.
_ = pipeline.run_query(entity_id='NEED-001', top_k=1)
print('listo — el proceso está caliente, las siguientes consultas salen en 1-2s')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

listo — el proceso está caliente, las siguientes consultas salen en 1-2s


## Celda de consulta — reemplazar el ID por el que dé el evaluador

In [2]:
QUERY_ID = 'NEED-013'  # <-- cambiar por el ID que pida el evaluador

out = pipeline.run_query(entity_id=QUERY_ID, top_k=5)
print(f"consulta: {out['source']['id']} ({out['source']['type']}, official={out['source']['official']})")
print(f"tiempo: {out['meta']['elapsed_seconds']}s\n")

for i, r in enumerate(out['results'], 1):
    rel = r['relevance']
    print(f"{i}. {r['target']['type']} {r['target']['id']}  score={rel['score']:.2f} ({rel['label']})")

consulta: NEED-013 (NEED, official=True)
tiempo: 1.644s

1. PRJ PRJ-097  score=0.67 (media)
2. THS THS-241  score=0.67 (media)
3. THS THS-246  score=0.67 (media)
4. THS THS-251  score=0.65 (media)
5. PRJ PRJ-102  score=0.63 (media)


## Explicación + evidencia del resultado #1 (para auditar en vivo)

In [3]:
top = out['results'][0]
print('EXPLICACION:')
print(' ', top['explanation'])
print()
print('EVIDENCIA (verificar a mano contra el archivo crudo):')
for ev in top['evidence']:
    print(f"  archivo: {ev['file']}")
    print(f"  registro: {ev['id']}  campo: {ev['field']}")
    print(f"  snippet citado: {ev['snippet']}")
    print()

EXPLICACION:
  PRJ-097 es un resultado de relevancia media (score=0.67) para NEED-013. Similitud semántica 0.87 entre el campo 'title' de NEED-013 y 'title' de PRJ-097. Comparte los términos de dominio: financial anomaly, fraude financiero (dominio=0.66). Señal de metodología no aplica: la fuente no expone un campo de metodología por diseño. Proximidad estructural en el grafo institucional: NEED-013 -[NEED.originating_unit(regex)]-> FAC-003 ; FAC-003 -[PRJ.faculty_id]-> PRJ-097 (estructural=0.24).

EVIDENCIA (verificar a mano contra el archivo crudo):
  archivo: 03_knowledge_needs/institutional_needs.csv
  registro: NEED-013  campo: title
  snippet citado: Detección de fraude financiero

  archivo: 03_knowledge_needs/projects.csv
  registro: PRJ-097  campo: title
  snippet citado: Análisis aplicado de fraude financiero para fortalecer financial anomaly

  archivo: domain_vocab.json
  registro: PRJ-097  campo: domain_terms
  snippet citado: financial anomaly, fraude financiero

  archiv

## Oportunidades generadas

In [4]:
for o in out['opportunities']:
    print(f"[{o['type']}] {o['opportunity']}")
    print(f"  razón: {o['reason']}")
    print(f"  prioridad: {o['priority']}  |  entidades: {o['related_entities']}")
    print()
if not out['opportunities']:
    print('(ninguna oportunidad disparó para esta consulta)')

[RESEARCH_CONTINUITY] Continuar la línea de trabajo de PRJ-097 para atender NEED-013.
  razón: Antecedente con estado ACTIVE y dominio afín (y método afín, cuando aplica).
  prioridad: media  |  entidades: ['NEED-013', 'PRJ-097', 'GRP-011']



## [PLUS] Si el evaluador prefiere subir un PDF en vez de dar un ID

Requiere `saberlink/plus/docling_intake.py` (feature opcional, ver README). El perfil extraído se trata como necesidad temporal — nunca se persiste en `institutional_needs.csv`.

In [5]:
# from saberlink.plus.docling_intake import pdf_to_temp_need
# profile = pdf_to_temp_need('ruta/al/pdf/del/evaluador.pdf')
# out_plus = pipeline.run_query(raw_text_profile=profile, top_k=5)
# print(out_plus['source'])  # official=False